In [6]:
import tifffile
import dask.array as da
import napari
from ome_zarr.io import parse_url
from ome_zarr.reader import Reader
from skimage import io

## Load image

In [9]:
fn = '/mnt/DATA/mouse_1.ome.zarr/'

In [10]:
# read the image data
reader = Reader(parse_url(fn))
# nodes may include images, labels etc
nodes = list(reader())
# first node will be the image pixel data
image_node = nodes[0]

dask_data = image_node.data

dask_data

version mismatch: detected: FormatV04, requested: FormatV05


[dask.array<from-zarr, shape=(3, 30089, 49350), dtype=>u2, chunksize=(1, 512, 512), chunktype=numpy.ndarray>,
 dask.array<from-zarr, shape=(3, 6017, 9870), dtype=>u2, chunksize=(1, 512, 512), chunktype=numpy.ndarray>,
 dask.array<from-zarr, shape=(3, 1203, 1974), dtype=>u2, chunksize=(1, 512, 512), chunktype=numpy.ndarray>]

In [11]:
dask_data[0]

dask.array<from-zarr, shape=(3, 30089, 49350), dtype=>u2, chunksize=(1, 512, 512), chunktype=numpy.ndarray>

## Load labels from JSON

In [16]:
labels_fn = '/mnt/DATA/Mouse_1_max_proj.geojson'

In [25]:
import geojson
with open(labels_fn) as f:
    gj = geojson.load(f)

In [30]:
dask_data[0].dtype

dtype('>u2')

## Convert labels to image array

In [31]:
import numpy as np
from tqdm.auto import tqdm
from shapely.geometry import Polygon
from shapely import geometry
from skimage.draw import polygon as draw_polygon

# ---- config ----
# gj = [...]  # your loaded json list of features
image_shape = dask_data[0].shape[1:]   # (Y, X) — set this to your image size
dtype = dask_data[0].dtype          # can change to uint8 if <255 objects

# ---- init blank segmentation ----
seg = np.zeros(image_shape, dtype=dtype)

# ---- iterate with tqdm ----
for i, feature in enumerate(tqdm(gj, desc="Rasterizing polygons", total=len(gj))):
    geom = feature["geometry"]
    if geom["type"] != "Polygon":
        continue
    coords = np.array(geom["coordinates"][0])  # [[x,y], ...]
    # separate x/y into skimage draw order (r=Y, c=X)
    rr, cc = draw_polygon(coords[:,1], coords[:,0], shape=image_shape)
    seg[rr, cc] = i + 1    # label index (1..N)

print("Done:", seg.shape, seg.dtype, "unique labels:", seg.max())


Rasterizing polygons:   0%|          | 0/17328 [00:00<?, ?it/s]

Done: (30089, 49350) >u2 unique labels: 17328


## Save out as Zarr labels layer

In [36]:
from pathlib import Path
import zarr, json
import numpy as np

try:
    from numcodecs import Blosc
    _compressor = Blosc(cname="zstd", clevel=5, shuffle=Blosc.SHUFFLE)  # good for labels
except Exception:
    # fallback if numcodecs is missing or zstd isn’t available
    _compressor = None
    
# --- inputs ---
store_path = Path("/mnt/DATA/mouse_1.ome.zarr")   # your OME-Zarr image
seg = seg.astype(np.uint32)                        # labels should be integer type
label_name = "seg"                                 # how it’ll appear in viewers

# --- open root and find the image group (typically the first array group in the store) ---
root = zarr.open_group(store=str(store_path), mode="r+")
# assume the image data live at the top level as multiscales[0]; adapt if yours is nested
img_ms = root.attrs["multiscales"][0]
img_group = root                                 # image group we’ll attach labels to

# read the image pyramid shapes to mirror label levels
img_dsets = img_ms["datasets"]                   # e.g. [{'path':'0', ...}, {'path':'1', ...}, ...]
img_shapes = [np.array(img_group[d["path"]]).shape for d in img_dsets]

# detect axes and pull out the last two (Y,X) sizes to match labels
axes = [ax["name"] for ax in img_ms.get("axes", [])] or ["c","y","x"]  # fallback
def yx_shape(shape):
    # works for (y,x), (c,y,x), (t,c,y,x), etc: take last two dims
    return tuple(shape[-2:])

yx_shapes = [yx_shape(s) for s in img_shapes]

# --- build (or reuse) the labels group structure per NGFF v0.5 ---
labels_parent = img_group.require_group("labels")
lab_group = labels_parent.require_group(label_name)

def write_level(g: zarr.Group, path: str, arr2d: np.ndarray, chunks=(512, 512)):
    ds = g.require_dataset(
        path,
        shape=arr2d.shape,
        dtype=arr2d.dtype,
        chunks=chunks,
        compressor=_compressor,   # << here
    )
    ds[:] = arr2d


# 1) write scale-0 (must match the *largest* XY of your image pyramid)
assert seg.shape[-2:] == yx_shapes[0], f"seg YX {seg.shape[-2:]} != image level-0 {yx_shapes[0]}"
write_level(lab_group, "0", seg)

# 2) OPTIONAL: make downsampled label levels to match the image pyramid
#    Use nearest-neighbour to preserve integer labels.
def down_nearest(lbl: np.ndarray, out_shape):
    # pure-numpy nearest resize for 2D labels
    y, x = lbl.shape
    oy, ox = out_shape
    yi = (np.arange(oy) * (y / oy)).astype(np.int64)
    xi = (np.arange(ox) * (x / ox)).astype(np.int64)
    return lbl[np.ix_(yi, xi)]

for lvl, target_yx in tqdm(enumerate(yx_shapes[1:], start=1), total = len(yx_shapes[1:])):
    write_level(lab_group, str(lvl), down_nearest(seg, target_yx))

# --- attach NGFF metadata for the labels group ---
# multiscales for the labels (same axes but only YX here)
lab_multiscales = [{
    "version": "0.5",
    "name": label_name,
    "axes": [ax for ax in img_ms.get("axes", []) if ax["name"] in ("y","x")] or [{"name":"y","type":"space"},{"name":"x","type":"space"}],
    "datasets": [{"path": str(i)} for i in range(len(yx_shapes))]
}]
lab_group.attrs["multiscales"] = lab_multiscales

# labels-specific metadata block (optional but helpful for viewers)
lab_group.attrs["image-label"] = {
    "version": "0.5",
    "name": label_name,
    # add a simple color map if you like; viewers can also choose random LUTs
    # "colors": [{"label-value": 1, "rgba": [255,0,0,255]}, {"label-value": 2, "rgba": [0,255,0,255]}]
}

# register this labels entry on the *image* group so readers discover it
labels_list = img_group.attrs.get("labels", [])
label_path = f"labels/{label_name}"
if not any(l.get("path")==label_path for l in labels_list if isinstance(l, dict)):
    labels_list.append({"path": label_path})
img_group.attrs["labels"] = labels_list

print(f"Done. Wrote labels at: {store_path}/labels/{label_name}")


/tmp/ipykernel_736512/3116510515.py:40: ZarrDeprecationWarning: Use Group.require_array instead.
  ds = g.require_dataset(


  0%|          | 0/2 [00:00<?, ?it/s]

Done. Wrote labels at: /mnt/DATA/mouse_1.ome.zarr/labels/seg


# Load all into napari

In [37]:
from pathlib import Path
import zarr
import dask.array as da

def open_ngff_pyramids(store_path, label_name="seg"):
    """
    Return (img_levels, lab_levels, img_axes, lab_axes)
      - img_levels: [dask.array, ...] for image multiscale (0 is highest res)
      - lab_levels: [dask.array, ...] for labels multiscale (may be single-scale)
      - img_axes / lab_axes: lists of axis dicts from NGFF metadata
    """
    store_path = Path(store_path)
    root = zarr.open_group(str(store_path), mode="r")

    # --- image pyramid ---
    img_ms = root.attrs["multiscales"][0]           # tolerate v0.4/v0.5
    img_axes = img_ms.get("axes", [{"name":"c"},{"name":"y"},{"name":"x"}])
    img_paths = [d["path"] for d in img_ms["datasets"]]
    img_levels = [da.from_zarr(str(store_path / p)) for p in img_paths]

    # --- labels pyramid (optional) ---
    lab_levels, lab_axes = [], [{"name":"y","type":"space"},{"name":"x","type":"space"}]
    lab_group_path = store_path / "labels" / label_name
    if lab_group_path.exists():
        lab_group = zarr.open_group(str(lab_group_path), mode="r")
        lab_ms = lab_group.attrs["multiscales"][0]
        lab_axes = lab_ms.get("axes", lab_axes)
        lab_paths = [d["path"] for d in lab_ms["datasets"]]
        lab_levels = [da.from_zarr(str(lab_group_path / p)) for p in lab_paths]
    else:
        print(f"[open_ngff_pyramids] No labels found at {lab_group_path}")

    return img_levels, lab_levels, img_axes, lab_axes

# --- example usage ---
img_levels, lab_levels, img_axes, lab_axes = open_ngff_pyramids("/mnt/DATA/mouse_1.ome.zarr", label_name="seg")

print("Image pyramid shapes:", [a.shape for a in img_levels])
print("Label pyramid shapes:", [a.shape for a in lab_levels])
# Access level-0 (highest res):
img0 = img_levels[0]
lab0 = lab_levels[0] if lab_levels else None


Image pyramid shapes: [(3, 30089, 49350), (3, 6017, 9870), (3, 1203, 1974)]
Label pyramid shapes: [(30089, 49350), (6017, 9870), (1203, 1974)]


In [38]:
viewer = napari.Viewer(title = 'loading baptistes labels')
viewer.add_image(img_levels, channel_axis=0,)# scale = (2.0, 0.1625, 0.1625))

[<Image layer 'Image' at 0x72b6c031eb90>,
 <Image layer 'Image [1]' at 0x72b6d8209090>,
 <Image layer 'Image [2]' at 0x72b67532c050>]

In [39]:
viewer.add_labels(lab_levels)

<Labels layer 'lab_levels' at 0x72b4d3864190>